# 02 — Intraday data and completed candles

Validates `qbs/trading_agent/market_context.py` and `session_calendar.py` on the
versioned fixture.

**Expected behaviour**
- only COMPLETED 15m/30m candles enter the snapshot; the daily series stops at the
  previous session;
- timestamps are tz-aware and align with the America/New_York session, across DST;
- indicators are finite where history allows and `null` (with a warning) where not;
- a missing or late candle makes the snapshot STALE/MISSING, so the engine sends no
  request;
- the fixture feed can never reveal a candle after the simulated clock.

**Modes.** `MOCK` (default) makes zero external LLM calls. `REAL_LLM` needs
`QBS_AGENT_TEST_MODE=REAL_LLM` **and** `QBS_AGENT_TEST_ALLOW_PAID=1`, prints a cost
estimate first, and stops at `QBS_AGENT_TEST_MAX_CALLS` (default 20).

**Isolation.** Every run uses a fresh temporary database (`tk.workspace()`); the
production config (`var/agent/config.json`) and log (`var/agent/agent.db`) are never
read or written, and nothing here can reach `qbs.live` or a broker.

**Dependencies.** `pip install -r requirements.txt jupyter` (MOCK). REAL_LLM also needs
the selected provider's package and key (see docs/TRADING_AGENT.md).

In [1]:
import os, sys, pathlib, json, logging
here = pathlib.Path.cwd()
ROOT = next(p for p in [here, *here.parents] if (p / "qbs" / "trading_agent").is_dir())
sys.path.insert(0, str(ROOT))
logging.basicConfig(level=logging.WARNING)

import pandas as pd
from qbs.trading_agent import testkit as tk

MODE = tk.test_mode()          # MOCK unless QBS_AGENT_TEST_MODE=REAL_LLM and QBS_AGENT_TEST_ALLOW_PAID=1
print("repo:", ROOT)
print("MODE:", MODE, "| paid-call cap:", tk.max_calls() if MODE == tk.REAL_LLM else 0)

repo: /home/user/qqq_boxx_strategies
MODE: MOCK | paid-call cap: 0


In [2]:
from datetime import date, timedelta, timezone
from qbs.trading_agent.market_context import MarketContextBuilder, FRESH, STALE, MISSING, completed
from qbs.trading_agent.session_calendar import (analysis_slots, session_bounds, due_slot,
                                                is_early_close, nyse_holidays, ET)
fx = tk.load_fixture()
day = fx.replay_date
check = tk.Check("02 intraday data")
print(fx.manifest["version"], fx.manifest["source"], "replay", day, "| real prices:",
      fx.manifest["real_prices"])
print(fx.manifest["scenario"])

v1 synthetic replay 2026-10-08 | real prices: False
TEAM breaks out above 200 around 11:15 ET and retests; MRVL (held, 9 shares) peaks near noon and loses VWAP after 14:00; TSM pulls back to ~294 then recovers above 302 late


## Fixture integrity: OHLC sanity, 15m ↔ 30m alignment, timezone

In [3]:
for sym in fx.symbols:
    b15, b30 = fx.bars[(sym, "15m")], fx.bars[(sym, "30m")]
    ok = ((b15.High >= b15[["Open", "Close"]].max(axis=1)) &
          (b15.Low <= b15[["Open", "Close"]].min(axis=1))).all()
    check(f"{sym} 15m OHLC consistent", bool(ok))
    check(f"{sym} index tz is America/New_York", str(b15.index.tz) == "America/New_York")
    inside = ((b15.index.time >= pd.Timestamp("09:30").time()) &
              (b15.index.time < pd.Timestamp("16:00").time())).all()
    check(f"{sym} bars inside regular hours", bool(inside))
    agg = b15.groupby(b15.index.floor("30min")).agg({"High": "max", "Low": "min", "Close": "last"})
    check(f"{sym} 30m bars = aggregated 15m", (agg.round(2).values == b30[["High", "Low", "Close"]].values).all())

## Completed-candle enforcement

At 10:22 ET the 10:15–10:30 candle is in progress. The fixture feed does not reveal it (it only knows its final values), and even a feed that DID serve it would be filtered by the builder — shown with `completed()` on the raw frame.

In [4]:
clock = tk.Clock(tk.et(day, 10, 22))
feed = tk.FixtureBars(fx, clock)
snap = MarketContextBuilder(feed).build("TEAM", tk.et(day, 10, 15))
check("snapshot FRESH at 10:15", snap.freshness == FRESH, snap.error or "")
check("latest completed candle ends 10:15",
      snap.data["latest_completed_15m_end"] == tk.et(day, 10, 15).isoformat())
check("newest raw 15m row is the 10:00 bar", snap.data["tf_15m"]["raw"]["rows"][-1][0].endswith("10:00"))
check("daily rows stop before the replay day",
      all(r[0] < day.isoformat() for r in snap.data["daily"]["raw"]["rows"]))
check("feed never revealed past the clock", feed.revealed_max <= clock())

raw = fx.bars[("TEAM", "15m")]
leaky = raw[raw.index <= clock()]                    # what a careless feed would hand over
kept = completed(leaky, "15m", tk.et(day, 10, 15))
check("builder drops the in-progress bar", len(leaky) - len(kept) == 1,
      f"{leaky.index[-1]:%H:%M} dropped")
print(json.dumps(snap.data["today"], indent=1))

{
 "open": 197.38,
 "high": 198.46,
 "low": 197.18,
 "last": 198.22,
 "volume": 204435,
 "vwap": 197.7,
 "change_vs_prev_close_pct": 0.62,
 "completed_15m_bars": 3
}


## Indicators

In [5]:
tf = snap.data["tf_15m"]
vals = {"ema9": tf["ema9"], "ema21": tf["ema21"], "rsi14": tf["rsi14"], "atr14": tf["atr14"],
        "macd": tf["macd"]["line"], "vwap": snap.data["today"]["vwap"],
        "daily_ema20": snap.data["daily"]["ema20"], "daily_sma50": snap.data["daily"]["sma50"]}
for k, v in vals.items():
    check(f"{k} computed", v is not None and v == v, str(v))
check("RSI in [0, 100]", 0 <= tf["rsi14"] <= 100)
t = fx.bars[("TEAM", "15m")]; t = t[t.index.date == day].head(3)
vw = ((t.High + t.Low + t.Close) / 3 * t.Volume).sum() / t.Volume.sum()
check("VWAP matches a hand computation", abs(vw - snap.data["today"]["vwap"]) < 0.01, f"{vw:.2f}")
# With too little history an indicator is null, never guessed:
short = tk.FixtureBars(fx, tk.Clock(tk.et(day, 9, 46)))
short.fixture = tk.Fixture(fx.version, fx.manifest,
                           {k: (v[v.index.date == day] if k[1] != "1d" else v) for k, v in fx.bars.items()})
s2 = MarketContextBuilder(short).build("TEAM", tk.et(day, 9, 45))
check("EMA21 null with one bar of history", s2.data["tf_15m"]["ema21"] is None)
check("...and a warning says so", any("EMA21" in w for w in s2.warnings))
pd.Series(vals)

ema9           197.830
ema21          198.140
rsi14           49.400
atr14            0.577
macd            -0.344
vwap           197.700
daily_ema20    192.250
daily_sma50    183.940
dtype: float64

## Missing and late candles

In [6]:
late = tk.FixtureBars(fx, tk.Clock(tk.et(day, 10, 31)),
                     withhold={("TEAM", tk.et(day, 10, 30).isoformat())})
s = MarketContextBuilder(late).build("TEAM", tk.et(day, 10, 30))
check("withheld candle -> STALE", s.freshness == STALE, s.error)
gone = tk.FixtureBars(fx, tk.Clock(tk.et(day, 10, 31)), missing={"TSM"})
s = MarketContextBuilder(gone).build("TSM", tk.et(day, 10, 30))
check("no bars -> MISSING", s.freshness == MISSING and not s.usable, s.error)

True

## Session boundaries, holidays, early closes and DST

In [7]:
slots = analysis_slots(day)
check("first slot 09:45, last 15:45", (slots[0].strftime("%H:%M"), slots[-1].strftime("%H:%M")) == ("09:45", "15:45"))
check("26 candles in a regular day, 25 analysed", len(slots) == 25)
check("no slot before the first candle completes", due_slot(tk.et(day, 9, 44)) is None)
check("Thanksgiving closed", analysis_slots(date(2026, 11, 26)) == [])
check("day after Thanksgiving ends 12:45", analysis_slots(date(2026, 11, 27))[-1].strftime("%H:%M") == "12:45")
winter = session_bounds(date(2026, 3, 6))[0].astimezone(timezone.utc).strftime("%H:%M")
summer = session_bounds(date(2026, 3, 9))[0].astimezone(timezone.utc).strftime("%H:%M")
check("DST: open moves 14:30 -> 13:30 UTC", (winter, summer) == ("14:30", "13:30"), f"{winter} / {summer}")
pd.DataFrame({"holiday_2026": [str(d) for d in nyse_holidays(2026)]}).T

,0,1,2,3,4,5,6,7,8,9
holiday_2026,2026-01-01,2026-01-19,2026-02-16,2026-04-03,2026-05-25,2026-06-19,2026-07-03,2026-09-07,2026-11-26,2026-12-25


## Result

In [8]:
res = check.frame(); display(res)
print(tk.save_results("02_intraday_data", res))
check.raise_if_failed()

,check,result,detail
0,TEAM 15m OHLC consistent,PASS,
1,TEAM index tz is America/New_York,PASS,
2,TEAM bars inside regular hours,PASS,
3,TEAM 30m bars = aggregated 15m,PASS,
4,MRVL 15m OHLC consistent,PASS,
5,MRVL index tz is America/New_York,PASS,
6,MRVL bars inside regular hours,PASS,
7,MRVL 30m bars = aggregated 15m,PASS,
8,TSM 15m OHLC consistent,PASS,
9,TSM index tz is America/New_York,PASS,


/tmp/claude-0/-home-user-qqq-boxx-strategies/d20be8e8-2747-505a-9165-041a5d5e0f58/scratchpad/nbres/02_intraday_data_20261010T092856Z.csv


**Troubleshooting**
- *STALE in production at every slot*: Yahoo publishes late — raise `data_delay_seconds`.
- *Indicator null*: not enough completed bars yet (early in the session) — expected.
- *30m mismatch*: a provider aligning 30m bars differently (e.g. 09:45 starts) — the
  builder still only uses completed bars, but the fixture check flags the difference.